In [4]:
import pandas as pd
import random

# --- Configuration ---
input_csv_file = '/data/primark/original_training_set.csv'
output_csv_file = '/data/primark/risk_training_set.csv'  # New CSV file for labeled training data

# arbitrary rules to simulate return risk probability (low, medium, high)
# The goal is to create a somewhat realistic distribution, not perfect accuracy.

def assign_simulated_risk(row):
    """
    Assigns a simulated return risk (low, medium, high) based on product attributes.
    Adjusted rules and thresholds for a less conservative distribution.
    """
    risk_score = 0

    # Rule 1: Higher price might slightly increase risk (e.g., more consideration, higher expectation)
    # Increased the impact of price
    if 'Full Price ($)' in row and pd.notna(row['Full Price ($)']):
        try:
            price = float(row['Full Price ($)'])
            if price > 100: # Lowered threshold and increased score
                risk_score += 3
            elif price > 50: # Lowered threshold and increased score
                risk_score += 2
            elif price > 25:
                risk_score += 1
        except ValueError:
            pass

    # Rule 2: Certain categories might have higher return rates (e.g., "Dresses", "Jeans" for fit issues)
    # Increased the impact of high-risk categories
    category_lower = str(row.get('Category', '')).lower() if pd.notna(row.get('Category')) else ''
    if 'dress' in category_lower or 'jean' in category_lower or 'swimwear' in category_lower or 'lingerie' in category_lower: # Added lingerie
        risk_score += 3 # Increased score
    elif 'shoe' in category_lower or 'pant' in category_lower: # Added shoes and pants
        risk_score += 2 # Increased score
    elif 'top' in category_lower or 'accessory' in category_lower:
        risk_score += 1 # Slightly increased score even for lower risk categories

    # Rule 3: Specific keywords in description might indicate fit issues or quality concerns
    # Increased the impact of problematic keywords
    description_lower = str(row.get('Description', '')).lower() if pd.notna(row.get('Description')) else ''
    if 'runs small' in description_lower or 'tight fit' in description_lower or 'see-through' in description_lower or 'delicate' in description_lower: # Added delicate
        risk_score += 4 # Increased score significantly
    elif 'true to size' in description_lower or 'comfortable' in description_lower or 'durable' in description_lower: # Added durable
        risk_score -= 2 # Increased negative impact

    # Rule 4: Number of SKUs available (fewer options might mean less choice, higher risk of wrong fit)
    # Slightly increased impact
    if 'SKUs Available' in row and pd.notna(row['SKUs Available']):
        try:
            skus_str = str(row['SKUs Available'])
            if '/' in skus_str:
                available_skus = int(skus_str.split('/')[0]) # Use available SKUs before the slash
                if available_skus < 3: # Lowered threshold
                    risk_score += 2 # Increased score
                elif available_skus < 6:
                    risk_score += 1
            else: # Handle cases where SKUs Available is just a number
                 available_skus = int(skus_str)
                 if available_skus < 3:
                    risk_score += 2
                 elif available_skus < 6:
                    risk_score += 1

        except ValueError:
            pass

    # Rule 5: Gender-specific clothing might have different return patterns (arbitrary example)
    # Slightly increased impact
    gender_lower = str(row.get('Gender', '')).lower() if pd.notna(row.get('Gender')) else ''
    if 'female' in gender_lower:
        risk_score += 2 # Increased score

    # Rule 6: Activewear might have lower return rates due to functional purpose
    # Increased negative impact
    activewear_lower = str(row.get('Activewear', '')).lower() if pd.notna(row.get('Activewear')) else ''
    if 'yes' in activewear_lower or 'true' in activewear_lower:
        risk_score -= 2 # Increased negative impact

    # Add a small random component to introduce variability
    risk_score += random.uniform(-1.5, 1.5) # Increased range of random variability

    # Map the cumulative risk score to 'low', 'medium', 'high'
    # Adjusted thresholds for a less conservative distribution
    if risk_score <= 2: # Lowered threshold for medium
        return 'low'
    elif risk_score <= 5: # Lowered threshold for high
        return 'medium'
    else:
        return 'high'

# --- Main Script ---
print(f"Attempting to load data from: {input_csv_file}")
try:
    df = pd.read_csv(input_csv_file)
    print(f"Successfully loaded '{input_csv_file}'. Shape: {df.shape}")
    print("Original columns:", df.columns.tolist())

except FileNotFoundError:
    print(f"Error: Input file not found at {input_csv_file}")
    exit()
except Exception as e:
    print(f"An error occurred while loading the CSV: {e}")
    exit()


# Apply the simulation function to create the new column
print("\nSimulating 'return_risk_probability' column...")
df['return_risk_probability'] = df.apply(assign_simulated_risk, axis=1)

print("\nNew DataFrame with 'return_risk_probability' column:")
print(df[['Full Price ($)', 'Category', 'Description', 'SKUs Available', 'return_risk_probability']].head(10))
print(f"\nDistribution of simulated risk levels:\n{df['return_risk_probability'].value_counts()}")

# Save the updated DataFrame to a new CSV file
try:
    df.to_csv(output_csv_file, index=False)
    print(f"\nSuccessfully saved the labeled training data to '{output_csv_file}'.")
except Exception as e:
    print(f"An error occurred while saving the CSV: {e}")

Attempting to load data from: /data/primark/original_training_set.csv
Error: Input file not found at /data/primark/original_training_set.csv

Simulating 'return_risk_probability' column...


NameError: name 'df' is not defined